# Lab 4 — Genetic Tests and Principal Component Analysis
**Sociogenomics 2025/2026 · University of Bologna · Prof. Nicola Barban**

This notebook covers the R analysis component of Lab 4. All PLINK commands are run separately in **[Google Cloud Shell](https://shell.cloud.google.com/)**.

---

## How to upload your data files

1. Run all PLINK commands in Cloud Shell and package the results:
   ```bash
   cd ~/Sociogenomics/Data
   zip lab4_results.zip hapmap3_summary.lmiss hapmap3_summary.imiss \
       hapmap3_summary.frq hapmap3_summary.hwe hapmap3_het.het \
       hapmap3_pca.eigenvec hapmap3_pca.eigenval pca_EUR.eigenvec \
       bmi_assoc.assoc.linear bmi_assoc_pca_corrected.assoc.linear \
       1kg_samples.txt
   ```
2. Download `lab4_results.zip` from Cloud Shell: click ⋮ → **Download** → type the path → Download.
3. Upload the zip here: click the **folder icon** in the left sidebar → **upload icon** → select `lab4_results.zip`.
4. Run the cell below to unzip.

In [ ]:
# Unzip the results archive uploaded via the Files panel
unzip("lab4_results.zip", overwrite = TRUE)
cat("Files available:\n")
print(list.files())

In [ ]:
# Install any missing packages (only needed the first time)
pkgs <- c("ggplot2", "data.table", "patchwork", "randomForest")
missing <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
if (length(missing) > 0) install.packages(missing, quiet = TRUE)

library(data.table)
library(ggplot2)
library(patchwork)

theme_set(theme_bw(base_size = 13))
cat("Libraries loaded.\n")

---
## Part I — Exploring Genetic Data

PLINK exports genotype quality statistics as plain text tables. We load and visualise them here.

### 1.1 SNP and individual missingness

In [ ]:
lmiss <- fread("hapmap3_summary.lmiss")
imiss <- fread("hapmap3_summary.imiss")

cat("SNPs in dataset:        ", nrow(lmiss), "\n")
cat("Individuals in dataset: ", nrow(imiss), "\n")
cat("SNP missingness range:  ", range(lmiss$F_MISS), "\n")
cat("Ind missingness range:  ", range(imiss$F_MISS), "\n")

p1 <- ggplot(lmiss, aes(x = F_MISS)) +
  geom_histogram(bins = 50, fill = "steelblue", colour = "white") +
  xlab("Per-SNP missing rate") + ylab("Number of SNPs") +
  ggtitle("SNP missingness")

p2 <- ggplot(imiss, aes(x = F_MISS)) +
  geom_histogram(bins = 40, fill = "coral", colour = "white") +
  xlab("Per-individual missing rate") + ylab("Number of individuals") +
  ggtitle("Individual missingness")

p1 + p2

### 1.2 Minor allele frequency (MAF) distribution

In [ ]:
frq <- fread("hapmap3_summary.frq")
cat("Mean MAF:  ", round(mean(frq$MAF), 4), "\n")
cat("Median MAF:", round(median(frq$MAF), 4), "\n")

ggplot(frq, aes(x = MAF)) +
  geom_histogram(bins = 50, fill = "darkgreen", colour = "white") +
  geom_vline(xintercept = 0.05, colour = "red",
             linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = 0.07, y = Inf, vjust = 2,
           label = "MAF = 0.05", colour = "red", size = 4) +
  xlab("Minor allele frequency") + ylab("Number of SNPs") +
  ggtitle("MAF distribution across all SNPs")

In [ ]:
# Count SNPs per MAF bin
frq[, bin := fcase(
  MAF < 0.01,               "< 0.01 (rare)",
  MAF < 0.05,               "0.01 – 0.05",
  MAF < 0.10,               "0.05 – 0.10",
  MAF < 0.20,               "0.10 – 0.20",
  default =                 ">= 0.20 (common)"
)]
print(frq[, .N, by = bin][order(bin)])

### 1.3 Hardy-Weinberg Equilibrium (HWE) distribution

SNPs deviating from HWE may indicate genotyping errors. The standard QC threshold is $p < 10^{-6}$.

In [ ]:
hwe     <- fread("hapmap3_summary.hwe")
hwe_all <- hwe[TEST == "ALL"]
hwe_all[, log10p := -log10(P)]

ggplot(hwe_all, aes(x = log10p)) +
  geom_histogram(bins = 60, fill = "purple", colour = "white") +
  geom_vline(xintercept = 6, colour = "red",
             linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = 6.3, y = Inf, vjust = 2,
           label = "p = 1e-6", colour = "red", size = 4) +
  xlab(expression(-log[10](p))) + ylab("Number of SNPs") +
  ggtitle("Distribution of HWE test statistics")

In [ ]:
cat("SNPs failing HWE (p < 1e-6):", sum(hwe_all$P < 1e-6, na.rm = TRUE), "\n")
cat("\nTop 10 most deviant SNPs:\n")
print(hwe_all[order(P)][1:10, .(SNP, CHR, `O(HET)`, `E(HET)`, P)])

### 1.4 Per-individual inbreeding coefficient

The inbreeding coefficient $F$ measures deviation from expected heterozygosity.
- $F < -0.15$: excess heterozygosity → possible sample contamination
- $F > +0.15$: deficit of heterozygosity → possible inbreeding or genotyping error

In [ ]:
het <- fread("hapmap3_het.het")

ggplot(het, aes(x = F)) +
  geom_histogram(bins = 50, fill = "orange", colour = "white") +
  geom_vline(xintercept = c(-0.15, 0.15), colour = "red",
             linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = -0.17, y = Inf, vjust = 2, hjust = 1,
           label = "-0.15", colour = "red", size = 4) +
  annotate("text", x =  0.17, y = Inf, vjust = 2, hjust = 0,
           label = "+0.15", colour = "red", size = 4) +
  xlab("Inbreeding coefficient F") + ylab("Number of individuals") +
  ggtitle("Per-individual inbreeding coefficient")

In [ ]:
outliers <- het[F < -0.15 | F > 0.15]
cat("Heterozygosity outliers:", nrow(outliers), "\n")
if (nrow(outliers) > 0) print(outliers[, .(FID, IID, F)])

---
## Part II — Principal Component Analysis (PCA)

PCA summarises genome-wide allele frequency variation into orthogonal axes. Individuals with similar ancestry cluster together in PC space.

> **PCA was computed in Cloud Shell with PLINK** (`--pca 20` on the LD-pruned SNP set). We load the results here.

### 2.1 Scree plot — variance explained by each PC

In [ ]:
eigenval        <- fread("hapmap3_pca.eigenval", header = FALSE, col.names = "eigenvalue")
eigenval[, PC  := seq_len(.N)]
eigenval[, pct := eigenvalue / sum(eigenvalue) * 100]

ggplot(eigenval, aes(x = PC, y = pct)) +
  geom_col(fill = "steelblue", colour = "white") +
  geom_line(aes(group = 1)) +
  geom_point(size = 2) +
  scale_x_continuous(breaks = 1:20) +
  xlab("Principal Component") + ylab("Variance explained (%)") +
  ggtitle("Scree plot")

In [ ]:
print(eigenval[, .(PC, pct = round(pct, 2))])

### 2.2 Load PCA scores and population labels

In [ ]:
pc_cols <- c("FID", "IID", paste0("PC", 1:20))
pca     <- fread("hapmap3_pca.eigenvec", header = FALSE, col.names = pc_cols)

geo <- fread("1kg_samples.txt", sep = "\t", header = TRUE)
setnames(geo, "Sample name", "IID")

data <- merge(
  pca,
  geo[, .(`IID`, `Population code`, `Population name`,
           `Superpopulation code`, `Superpopulation name`)],
  by = "IID"
)

cat("Individuals with population labels:", nrow(data), "\n")
print(table(data$`Superpopulation name`))

### 2.3 PC1 vs PC2 — coloured by superpopulation

You should see five clearly separated clusters:
**AFR** (African), **EUR** (European), **EAS** (East Asian), **SAS** (South Asian), **AMR** (Admixed American).

In [ ]:
ggplot(data, aes(x = PC1, y = PC2, colour = `Superpopulation name`)) +
  geom_point(alpha = 0.7, size = 1.5) +
  xlab("PC1") + ylab("PC2") +
  labs(colour = "Superpopulation",
       title  = "PCA — continental ancestry (PC1 vs PC2)")

### 2.4 PC1 vs PC2 — coloured by sub-population

In [ ]:
ggplot(data, aes(x = PC1, y = PC2, colour = `Population name`)) +
  geom_point(alpha = 0.7, size = 1.5) +
  xlab("PC1") + ylab("PC2") +
  labs(colour = "Population",
       title  = "PCA — sub-population (PC1 vs PC2)") +
  theme(legend.text = element_text(size = 7))

### 2.5 PC1 vs PC3

In [ ]:
ggplot(data, aes(x = PC1, y = PC3, colour = `Superpopulation name`)) +
  geom_point(alpha = 0.7, size = 1.5) +
  xlab("PC1") + ylab("PC3") +
  labs(colour = "Superpopulation", title = "PC1 vs PC3")

### 2.6 Within-European PCA

Running PCA only within Europeans reveals finer-scale structure: Northern (Finnish, British) vs Southern (Iberian, Tuscan) Europeans.

In [ ]:
pc_cols_eur <- c("FID", "IID", paste0("PC", 1:10))
pca_eur     <- fread("pca_EUR.eigenvec", header = FALSE, col.names = pc_cols_eur)
data_eur    <- merge(pca_eur, geo[, .(`IID`, `Population name`)], by = "IID")

ggplot(data_eur, aes(x = PC1, y = PC2, colour = `Population name`)) +
  geom_point(alpha = 0.8, size = 2) +
  xlab("PC1") + ylab("PC2") +
  labs(colour = "European population",
       title  = "PCA within European populations")

### 2.7 Genomic inflation factor λ_GC

$$\lambda_{GC} = \frac{\text{median}(\chi^2_{\text{observed}})}{0.4549}$$

A value close to **1.0** indicates no inflation from population stratification. We compare λ before and after adding PCs as covariates.

In [ ]:
compute_lambda <- function(pvals) {
  pvals <- pvals[!is.na(pvals)]
  chisq <- qchisq(pvals, df = 1, lower.tail = FALSE)
  median(chisq) / 0.4549
}

# Uncorrected GWAS
res_raw  <- fread("bmi_assoc.assoc.linear")
lam_raw  <- compute_lambda(res_raw$P)
cat("Lambda (uncorrected): ", round(lam_raw, 3), "\n")

# PC-corrected GWAS (keep only the ADD test row per SNP)
res_corr <- fread("bmi_assoc_pca_corrected.assoc.linear")
res_corr <- res_corr[TEST == "ADD"]
lam_corr <- compute_lambda(res_corr$P)
cat("Lambda (PC-corrected):", round(lam_corr, 3), "\n")

---
## Part III — Detecting and Removing Population Outliers

In a study designed to be homogeneous (e.g., European-only), we identify individuals who cluster far from the main group in PC space and flag them for removal.

In [ ]:
# Compute EUR centroid and standard deviations
eur_mean_pc1 <- mean(data[`Superpopulation code` == "EUR", PC1])
eur_mean_pc2 <- mean(data[`Superpopulation code` == "EUR", PC2])
eur_sd_pc1   <- sd(data[`Superpopulation code`   == "EUR", PC1])
eur_sd_pc2   <- sd(data[`Superpopulation code`   == "EUR", PC2])

cat("EUR centroid: PC1 =", round(eur_mean_pc1, 4),
    ", PC2 =", round(eur_mean_pc2, 4), "\n")

# Flag individuals within 3 SD of the EUR centroid on PC1 and PC2
data[, eur_like := abs(PC1 - eur_mean_pc1) < 3 * eur_sd_pc1 &
                   abs(PC2 - eur_mean_pc2) < 3 * eur_sd_pc2]

cat("Individuals within 3 SD of EUR centroid:", sum(data$eur_like), "\n")

In [ ]:
ggplot(data, aes(x = PC1, y = PC2,
                 colour = `Superpopulation code`,
                 shape  = eur_like)) +
  geom_point(alpha = 0.7, size = 1.5) +
  scale_shape_manual(values = c(4, 16),
                     labels = c("Excluded", "EUR-like (kept)")) +
  labs(colour = "Superpopulation",
       shape  = "Selection",
       title  = "EUR-like individuals (within 3 SD of EUR centroid)")

In [ ]:
# Save the EUR-like sample list for PLINK
# Download this file from the Files panel (right-click → Download)
# then upload it to Cloud Shell and run:
#   plink --bfile hapmap3_qc --keep samples_EUR_like.txt --make-bed --out hapmap3_EUR
eur_keep <- data[eur_like == TRUE, .(FID, IID)]
fwrite(eur_keep, "samples_EUR_like.txt", sep = " ", col.names = FALSE)
cat("Saved", nrow(eur_keep), "EUR-like individuals to samples_EUR_like.txt\n")

---
## Part IV — Ancestry Prediction from PCA

PCA scores can predict the ancestry of individuals of unknown origin by comparing their PC coordinates to a labelled reference panel. This is the principle behind commercial ancestry tests (23andMe, AncestryDNA, etc.).

We use the 1000 Genomes superpopulation labels as reference and train:
1. **k-Nearest Neighbours (k-NN)** — assigns the majority label among the k closest individuals in PC space
2. **Random Forest** — an ensemble of decision trees that captures non-linear boundaries

### 4.1 Prepare features and train/test split

In [ ]:
library(class)

# Features: top 10 PCs
pc_features <- paste0("PC", 1:10)
X <- as.matrix(data[, ..pc_features])
y <- data$`Superpopulation code`

cat("Total individuals:", nrow(X), "\n")
print(table(y))

# 80/20 train/test split (stratified by hand)
set.seed(42)
n         <- nrow(data)
train_idx <- sample(n, size = floor(0.8 * n), replace = FALSE)
test_idx  <- setdiff(seq_len(n), train_idx)

train_X <- X[train_idx, ]; test_X <- X[test_idx, ]
train_y <- y[train_idx];   test_y <- y[test_idx]

cat("\nTraining set:", nrow(train_X), "individuals\n")
cat("Test set:    ", nrow(test_X),  "individuals\n")

### 4.2 k-NN classifier (k = 5)

In [ ]:
predicted <- knn(train = train_X, test = test_X, cl = train_y, k = 5)

# Confusion matrix
conf_mat <- table(Predicted = predicted, True = test_y)
print(conf_mat)

# Overall accuracy
accuracy <- sum(diag(conf_mat)) / sum(conf_mat)
cat("\nOverall accuracy:", round(accuracy * 100, 1), "%\n")

# Per-population accuracy
per_pop <- diag(conf_mat) / colSums(conf_mat)
cat("\nPer-population accuracy:\n")
print(round(per_pop * 100, 1))

### 4.3 Effect of k on accuracy

In [ ]:
k_values   <- c(1, 3, 5, 10, 20)
accuracies <- numeric(length(k_values))

for (i in seq_along(k_values)) {
  pred          <- knn(train_X, test_X, train_y, k = k_values[i])
  cm            <- table(pred, test_y)
  accuracies[i] <- sum(diag(cm)) / sum(cm) * 100
}

results <- data.frame(k = k_values, accuracy = round(accuracies, 1))
print(results)

ggplot(results, aes(x = k, y = accuracy)) +
  geom_line() +
  geom_point(size = 3, colour = "steelblue") +
  xlab("k (number of neighbours)") +
  ylab("Classification accuracy (%)") +
  ggtitle("k-NN ancestry classification accuracy") +
  ylim(90, 100)

### 4.4 Visualise predictions in PC space

Misclassified individuals are shown as **×**. They are most common among **AMR** (Admixed American) individuals, who sit genetically between other groups.

In [ ]:
# Predict ancestry for ALL individuals
predicted_all <- knn(train = train_X, test = X, cl = train_y, k = 5)

data[, predicted := as.character(predicted_all)]
data[, correct   := predicted == `Superpopulation code`]

ggplot(data, aes(x = PC1, y = PC2,
                 colour = `Superpopulation code`,
                 shape  = correct)) +
  geom_point(alpha = 0.7, size = 1.8) +
  scale_shape_manual(values = c(4, 16),
                     labels = c("Misclassified", "Correct")) +
  labs(colour = "True superpopulation",
       shape  = "Classification",
       title  = "k-NN ancestry predictions (k = 5)")

In [ ]:
misclass <- data[correct == FALSE,
                 .(`IID`, `Superpopulation code`, predicted, PC1, PC2)]
cat("Misclassified individuals:", nrow(misclass), "\n")
print(head(misclass, 20))

### 4.5 Random Forest classifier (bonus)

Random Forests capture non-linear decision boundaries and typically outperform k-NN. The variable importance plot shows which PCs matter most for separating populations.

In [ ]:
library(randomForest)

train_df <- data.frame(train_X, superpop = factor(train_y))
test_df  <- data.frame(test_X)

set.seed(42)
rf_model <- randomForest(superpop ~ ., data = train_df,
                         ntree = 500, importance = TRUE)

rf_pred <- predict(rf_model, newdata = test_df)
rf_cm   <- table(Predicted = rf_pred, True = test_y)
rf_acc  <- sum(diag(rf_cm)) / sum(rf_cm) * 100

cat("Random Forest accuracy:", round(rf_acc, 1), "%\n")
cat("k-NN accuracy (k=5):   ", round(accuracy * 100, 1), "%\n\n")
print(rf_cm)

In [ ]:
imp <- data.frame(
  PC          = rownames(importance(rf_model)),
  MeanDecGini = importance(rf_model)[, "MeanDecreaseGini"]
)

ggplot(imp, aes(x = reorder(PC, MeanDecGini), y = MeanDecGini)) +
  geom_col(fill = "steelblue") +
  coord_flip() +
  xlab("Principal Component") +
  ylab("Mean Decrease in Gini") +
  ggtitle("PC importance for ancestry classification\n(Random Forest)")

---
## Exercises

**Exercise 1 — Genotype quality**
1. How many SNPs deviate significantly from HWE at $p < 10^{-6}$? What could cause this?
2. Do the most deviant SNPs show excess or deficit heterozygosity?

**Exercise 2 — PCA**
1. How many PCs are needed to capture the main axes of variation? Where does the scree plot flatten?
2. Which superpopulations are most separated along PC1? Along PC2?
3. Within Europeans, which populations are most separated? What historical events might explain this?
4. How does $\lambda_{GC}$ change before and after adding 10 PCs as covariates?

**Exercise 3 — Ancestry prediction**
1. What is the k-NN accuracy for $k = 5$? Which superpopulation is hardest to classify and why?
2. Try using only PC1 and PC2 (`pc_features <- c("PC1", "PC2")`). How much does accuracy drop?
3. Look at the misclassified individuals. Which true superpopulation do they belong to, and which are they assigned to?
4. Compare k-NN vs Random Forest accuracy. Which performs better?